In [1]:
# Simplified Job Search Script - Individual Job Links Only
import warnings
warnings.filterwarnings('ignore')

from crewai import Agent, Task, Crew
import os
from dotenv import load_dotenv
from crewai_tools import (
    ScrapeWebsiteTool,
    SerperDevTool
)

load_dotenv()

openai_api_key = os.getenv('OPENAI_API_KEY')
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = os.getenv('SERPER_API_KEY', '')

# Initialize tools
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

# ===== ACTIVE AGENT - JOB LINKS DISCOVERY =====

# Job Links Discovery Agent (Enhanced for Individual Job URLs)
job_links_finder = Agent(
    role="Individual Job Links Finder",
    goal="Find and extract SPECIFIC individual job posting URLs (not category pages) from Indian job platforms",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "You are an expert at finding individual job posting URLs, not category pages. "
        "You know the difference between a category page like 'naukri.com/data-science-jobs' "
        "and actual job postings like 'naukri.com/job-listings/data-scientist-company-name-12345678'. "
        "Your specialty is navigating Indian job platforms to extract direct links to specific "
        "job postings that candidates can immediately apply to. You focus on getting job IDs, "
        "company-specific URLs, and individual posting links rather than search result pages."
    )
)

# ===== ACTIVE TASK - INDIVIDUAL JOB LINKS EXTRACTION =====

individual_job_links_task = Task(
    description=(
        "Search for INDIVIDUAL job posting URLs (not category pages) on these specific platforms:\n\n"
        "IMPORTANT: Do NOT return category/search pages like:\n"
        "❌ naukri.com/data-science-jobs\n"
        "❌ linkedin.com/jobs/data-science-jobs\n"
        "❌ wellfound.com/role/l/data-science/india\n\n"
        "Instead, find SPECIFIC job posting URLs like:\n"
        "✅ naukri.com/job-listings/data-scientist-xyz-company-bangalore-3-to-8-years-12345678\n"
        "✅ linkedin.com/jobs/view/1234567890\n"
        "✅ wellfound.com/jobs/1234567-data-scientist-at-startup-name\n\n"
        "TARGET PLATFORMS:\n"
        "1. Naukri.com - Find individual job posting URLs with job IDs\n"
        "2. LinkedIn Jobs India - Get specific 'linkedin.com/jobs/view/[JOB_ID]' URLs\n"
        "3. Instahyre - Extract individual job posting links\n"
        "4. Hirist - Get specific job posting URLs\n"
        "5. AngelList/Wellfound India - Find individual startup job posting URLs\n"
        "6. Company career pages - Direct job posting links from company websites\n\n"
        "SEARCH CRITERIA:\n"
        "- Job titles: {target_job_titles}\n"
        "- Locations: India (Bangalore, Mumbai, Delhi, Hyderabad, Chennai, Pune, Remote India)\n"
        "- Extract up to {max_jobs} INDIVIDUAL job posting URLs\n\n"
        "EXTRACTION METHOD:\n"
        "1. Search for jobs on each platform\n"
        "2. Navigate to actual search results\n"
        "3. Extract individual job posting URLs (with job IDs/unique identifiers)\n"
        "4. Verify each URL leads to a specific job posting, not a category page\n"
        "5. Include job title, company name, location, and direct apply URL for each"
    ),
    expected_output=(
        "A comprehensive list of INDIVIDUAL job posting URLs organized by platform:\n"
        "Format for each job:\n"
        "Platform: [Platform Name]\n"
        "Job Title: [Exact Job Title]\n"
        "Company: [Company Name]\n"
        "Location: [City, India]\n"
        "Direct Job URL: [Complete individual job posting URL]\n"
        "Quick Requirements: [Key requirements summary]\n"
        "---\n\n"
        "Ensure each URL is a direct link to an individual job posting that can be immediately applied to."
    ),
    agent=job_links_finder,
    output_file="individual_job_links.md"
)

# ===== SIMPLIFIED CREW WITH ONLY JOB DISCOVERY =====
job_links_crew = Crew(
    agents=[job_links_finder],
    tasks=[individual_job_links_task],
    verbose=True
)

# ===== EXECUTION FUNCTION =====
def find_individual_job_links(target_job_titles, max_jobs=25):
    """
    Find individual job posting URLs from Indian job platforms
    
    Args:
        target_job_titles: List of job titles to search for
        max_jobs: Maximum number of individual job links to find
    
    Returns:
        List of individual job posting URLs with details
    """
    
    job_search_inputs = {
        'target_job_titles': target_job_titles,
        'max_jobs': max_jobs
    }
    
    print(f"🔍 Searching for individual job posting URLs")
    print(f"🎯 Targeting {len(target_job_titles)} job title types")
    print(f"📊 Looking for up to {max_jobs} individual job links")
    print(f"🌐 Platforms: Naukri, LinkedIn India, Instahyre, Hirist, AngelList, Company pages")
    
    # Execute the simplified crew
    result = job_links_crew.kickoff(inputs=job_search_inputs)
    
    print("✅ Individual job links discovery completed!")
    print("📄 Generated file: individual_job_links.md")
    
    return result

# ===== EXAMPLE USAGE =====
if __name__ == "__main__":
    # Job titles to search for
    job_titles = [
        "Data Scientist",
        "Machine Learning Engineer",
        "Senior Data Scientist",
        "AI Engineer",
        "Data Engineer",
        "MLOps Engineer",
        "Analytics Engineer"
    ]
    
    # Run simplified job links discovery
    result = find_individual_job_links(
        target_job_titles=job_titles,
        max_jobs=25
    )


# ===== COMMENTED OUT - ALL OTHER AGENTS AND TASKS =====

# # 2. Job Links Aggregator Agent (COMMENTED)
# # job_links_aggregator = Agent(
# #     role="Job Links Collection Specialist",
# #     goal="Collect and organize all job links from multiple sources",
# #     tools=[search_tool, scrape_tool],
# #     verbose=True,
# #     backstory=("Job aggregation specialist...")
# # )

# # 3. Market Intelligence Agent (COMMENTED)
# # india_market_researcher = Agent(
# #     role="India Tech Market Intelligence Analyst",
# #     goal="Research Indian tech market trends",
# #     tools=[search_tool, scrape_tool],
# #     verbose=True,
# #     backstory=("Market research specialist...")
# # )

# # 4. Company Deep Dive Agent (COMMENTED)
# # india_company_researcher = Agent(
# #     role="India Company Culture & Background Researcher",
# #     goal="Research companies operating in India",
# #     tools=[search_tool, scrape_tool],
# #     verbose=True,
# #     backstory=("Company research specialist...")
# # )

# # 5. Personal Profiler (COMMENTED)
# # profiler = Agent(
# #     role="Personal Profiler for Engineers",
# #     goal="Research job applicants",
# #     tools=[scrape_tool, search_tool, read_resume, semantic_search_resume],
# #     verbose=True,
# #     backstory=("Personal profiling specialist...")
# # )

# # 6. Resume Strategist (COMMENTED)
# # resume_strategist = Agent(
# #     role="Resume Strategist for Engineers",
# #     goal="Create optimized resumes",
# #     tools=[scrape_tool, search_tool, read_resume, semantic_search_resume],
# #     verbose=True,
# #     backstory=("Resume optimization specialist...")
# # )

# # 7. Interview Preparer (COMMENTED)
# # interview_preparer = Agent(
# #     role="Engineering Interview Preparer",
# #     goal="Create interview preparation materials",
# #     tools=[scrape_tool, search_tool, read_resume, semantic_search_resume],
# #     verbose=True,
# #     backstory=("Interview preparation specialist...")
# # )

# # 8. Skills Analyzer (COMMENTED)
# # skills_analyzer = Agent(
# #     role="Skills Gap Analysis Expert",
# #     goal="Identify skill gaps",
# #     tools=[read_resume, semantic_search_resume, search_tool],
# #     verbose=True,
# #     backstory=("Skills analysis specialist...")
# # )

# # 9. Cover Letter Writer (COMMENTED)
# # cover_letter_writer = Agent(
# #     role="Cover Letter Specialist",
# #     goal="Create compelling cover letters",
# #     tools=[read_resume, semantic_search_resume, scrape_tool],
# #     verbose=True,
# #     backstory=("Cover letter specialist...")
# # )

# # 10. Application Strategist (COMMENTED)
# # application_strategist = Agent(
# #     role="Application Strategy Coordinator",
# #     goal="Develop strategic approaches",
# #     tools=[search_tool, scrape_tool],
# #     verbose=True,
# #     backstory=("Application strategy specialist...")
# # )

# # 11. Network Analyzer (COMMENTED)
# # network_analyzer = Agent(
# #     role="Professional Network Analyst",
# #     goal="Identify networking opportunities",
# #     tools=[search_tool, scrape_tool],
# #     verbose=True,
# #     backstory=("Networking specialist...")
# # )

# # 12. Application Tracker (COMMENTED)
# # application_tracker = Agent(
# #     role="Application Progress Manager",
# #     goal="Track application progress",
# #     tools=[search_tool],
# #     verbose=True,
# #     backstory=("Application tracking specialist...")
# # )

# # ===== COMMENTED OUT TASKS =====

# # Task 2: Job Links Display (COMMENTED)
# # job_links_display_task = Task(
# #     description="Create organized display of job links...",
# #     expected_output="Formatted job links...",
# #     context=[individual_job_links_task],
# #     agent=job_links_aggregator,
# #     output_file="job_links_directory.md"
# # )

# # All other tasks (COMMENTED)
# # market_research_task = Task(...)
# # company_research_task = Task(...)
# # profile_task = Task(...)
# # skills_gap_task = Task(...)
# # job_analysis_task = Task(...)
# # resume_task = Task(...)
# # cover_letter_task = Task(...)
# # application_strategy_task = Task(...)
# # network_analysis_task = Task(...)
# # interview_task = Task(...)
# # tracking_task = Task(...)

🔍 Searching for individual job posting URLs
🎯 Targeting 7 job title types
📊 Looking for up to 25 individual job links
🌐 Platforms: Naukri, LinkedIn India, Instahyre, Hirist, AngelList, Company pages
 [DEBUG]: == Working Agent: Individual Job Links Finder
 [INFO]: == Starting Task: Search for INDIVIDUAL job posting URLs (not category pages) on these specific platforms:

IMPORTANT: Do NOT return category/search pages like:
❌ naukri.com/data-science-jobs
❌ linkedin.com/jobs/data-science-jobs
❌ wellfound.com/role/l/data-science/india

Instead, find SPECIFIC job posting URLs like:
✅ naukri.com/job-listings/data-scientist-xyz-company-bangalore-3-to-8-years-12345678
✅ linkedin.com/jobs/view/1234567890
✅ wellfound.com/jobs/1234567-data-scientist-at-startup-name

TARGET PLATFORMS:
1. Naukri.com - Find individual job posting URLs with job IDs
2. LinkedIn Jobs India - Get specific 'linkedin.com/jobs/view/[JOB_ID]' URLs
3. Instahyre - Extract individual job posting links
4. Hirist - Get specific j